In [1]:
# ================================
# exp24_elasticnet_hyperensemble
# ElasticNet multi-hyperparameter ensemble
# ================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ================================
# Metric diagnostics tool
# ================================

def metric_diagnostics(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    y_true_log = np.log1p(y_true)
    y_pred_log = np.log1p(np.maximum(y_pred,0))
    rmsle = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    nrmse_mean = rmse / np.mean(y_true)
    nrmse_range = rmse / (np.max(y_true) - np.min(y_true))

    print("\nMetric diagnostics")
    print("------------------")
    print("RMSE:", rmse)
    print("MAE:", mae)
    print("RMSLE:", rmsle)
    print("NRMSE (mean):", nrmse_mean)
    print("NRMSE (range):", nrmse_range)


# ================================
# Load data
# ================================

train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

target = "含水率"
id_col = "sample number"

spectral_cols = [
    c for c in train.columns
    if c not in ["sample number","species number","樹種","含水率"]
]

X = train[spectral_cols]
y = train[target]

X_test = test[spectral_cols]


# ================================
# ElasticNet configurations
# ================================

elastic_configs = [
    (0.0005, 0.9),
    (0.001, 0.9),
    (0.002, 0.9),
    (0.001, 0.8)
]


# ================================
# KFold
# ================================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))


# ================================
# Training loop
# ================================

for alpha, l1_ratio in elastic_configs:

    print(f"\nTraining ElasticNet alpha={alpha}, l1_ratio={l1_ratio}")

    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=50000,
            random_state=42
        ))
    ])

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)

        pred_val = model.predict(X_val)

        oof[val_idx] = pred_val

        test_pred += model.predict(X_test) / kf.n_splits

    metric_diagnostics(y, oof)

    oof_predictions += oof / len(elastic_configs)
    test_predictions += test_pred / len(elastic_configs)


# ================================
# Final diagnostics
# ================================

print("\nFinal Ensemble Performance")
metric_diagnostics(y, oof_predictions)


# ================================
# Save submission
# ================================

os.makedirs("../submissions", exist_ok=True)

submission = pd.DataFrame({
    id_col: test[id_col],
    target: test_predictions
})

output_path = "../submissions/exp24_elasticnet_hyperensemble.csv"

submission.to_csv(output_path, index=False, header=False)

print("\nSubmission saved:", output_path)


Training ElasticNet alpha=0.0005, l1_ratio=0.9


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.250e+05, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.214e+05, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 15.382253195763427
MAE: 10.067650279221882
RMSLE: 0.5683384833545684
NRMSE (mean): 0.3080290403815544
NRMSE (range): 0.05166316232779154

Training ElasticNet alpha=0.001, l1_ratio=0.9


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.304e+05, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.251e+05, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 14.494829716542478
MAE: 9.604825029024962
RMSLE: 0.5523004339230942
NRMSE (mean): 0.2902584186632891
NRMSE (range): 0.04868264298013754

Training ElasticNet alpha=0.002, l1_ratio=0.9


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.345e+05, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.267e+05, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 13.812865401169663
MAE: 9.026110579572405
RMSLE: 0.4962579289325616
NRMSE (mean): 0.2766021089559045
NRMSE (range): 0.046392183144476365

Training ElasticNet alpha=0.001, l1_ratio=0.8


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.374e+05, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.298e+05, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 14.014126861656681
MAE: 9.177965367549106
RMSLE: 0.52194876577913
NRMSE (mean): 0.28063236211521936
NRMSE (range): 0.04706814416079437

Final Ensemble Performance

Metric diagnostics
------------------
RMSE: 14.19880295791305
MAE: 9.339710455032886
RMSLE: 0.5112599133309713
NRMSE (mean): 0.2843304939810326
NRMSE (range): 0.04768840122050676

Submission saved: ../submissions/exp24_elasticnet_hyperensemble.csv


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.280e+05, tolerance: 2.606e+02
  model = cd_fast.enet_coordinate_descent(
